In [1]:
import ROOT as rt
import ctypes
file = rt.TFile("e21062_44S.root")
dtimeN = file.Get("dtime")

/opt/anaconda3/envs/Sulfur/lib/python3.11/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /opt/anaconda3/envs/Sulfur/include/site/python3.11); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


In [2]:
#44SGated
def decay_parent(x, par):
    """
    a = amplitude
    b = decay constant (lambda)
    bkgrd = constant background
    """
    a = par[0]
    b = par[1]
    bkgrd = par[2]
    t = x[0]

    return a * rt.TMath.Exp(-b * t) + bkgrd

def Decayp(hist, fitgraph, X0, bounds):
    
    MaxX = hist.GetXaxis().GetXmax();
    #MinX = hist.GetXaxis().GetXmin();
    # Axis range
    MinX = 1
    #MaxX = 
    print("Min X: ", MinX)
    print("Max X: ", MaxX)
    
    # Canvas
    #fitgraph = rt.TCanvas("fits", "fits", 800, 600)
    fitgraph.Clear()
    
    # Define TF1
    decayp = rt.TF1("decayp", decay_parent, MinX, MaxX, len(X0))

    decayp.SetParName(0, "source #")
    decayp.SetParName(1, "Source lambda")
    decayp.SetParName(2, "shift")


    for i in range(0,len(X0)):# intial conditions
        decayp.SetParameter(i, X0[i])

    for i in range(0, len(bounds)): #Fit bounds
        decayp.SetParLimits(i, bounds[i][0], bounds[i][1]) 

    # Print initial parameters
    for i in range(decayp.GetNpar()):
        print(f"Initial p{i} = {decayp.GetParameter(i)}")

    # Perform fit
    hist.Fit("decayp", "RS", "", MinX, MaxX)
    #hist.Print("V")
    
    # Print parameter limits
    for i in range(decayp.GetNpar()):
        par_min = ctypes.c_double()
        par_max = ctypes.c_double()
        decayp.GetParLimits(i, par_min, par_max)
        print(f"Parameter {i} ({decayp.GetParName(i)}) limits: [{par_min}, {par_max}]")

    # Half-life calculation
    lambda_fit = decayp.GetParameter(1)
    lambda_err = decayp.GetParError(1)

    decayhl = rt.TMath.Log(2) / lambda_fit
    decay_error = (rt.TMath.Log(2) / (lambda_fit**2)) * lambda_err

    print(f"Decay: {decayhl:.6f} ms ± {decay_error:.6f} ms\n")

    # Chi2 / NDF
    chi2 = decayp.GetChisquare()
    ndf = decayp.GetNDF()
    print(f"Chi^2/NDF: {chi2/ndf:.6f}\n")

    # Constant background function
    bckgrnd = rt.TF1("bckgrnd", "[0]", MinX, MaxX)
    bckgrnd.SetParameter(0, decayp.GetParameter(2))
    bckgrnd.SetLineColor(rt.kBlack)

    # Legend
    legend = rt.TLegend(0.7, 0.4, 0.9, 0.7)
    legend.AddEntry(hist, "Data", "E")
    legend.AddEntry(decayp, "Fit", "l")
    legend.AddEntry(bckgrnd, "backgrnd", "l")

    # Styling
    hist.SetTitle("Decay of 44S Gated on 329keV(Neutron);Time (ms);Counts")
    hist.GetXaxis().CenterTitle(True)
    hist.GetYaxis().CenterTitle(True)
    hist.SetLineColor(rt.kBlue)
    decayp.SetLineColor(rt.kRed)

    # Draw
    hist.Draw("E")
    decayp.Draw("same")
    bckgrnd.Draw("same")
    fitgraph.Update()
    fitgraph.Draw()
    hist.Print("V")
    # legend.Draw()  # Uncomment if desired

    #fitgraph.SaveAs("GammaNTest.png")

In [3]:
bounds = [[0,800], # source
          [0,1],   #lamb
          [0,700]] #background shift

X0 = [700,
      .005,
      500]
gamma_canvas = rt.TCanvas("gamma_canvas_1", "gamma_329KeV")
gamma_329 = dtimeN.ProjectionX("Decay_Curve_329KeV",327,331)
#gamma_329.Draw()
Decayp(gamma_329, gamma_canvas,X0,bounds)
#gamma_canvas.Update()
#gamma_canvas.Draw()

Min X:  1
Max X:  1000.0
Initial p0 = 700.0
Initial p1 = 0.005
Initial p2 = 500.0
Parameter 0 (source #) limits: [c_double(0.0), c_double(800.0)]
Parameter 1 (Source lambda) limits: [c_double(0.0), c_double(1.0)]
Parameter 2 (shift) limits: [c_double(0.0), c_double(700.0)]
Decay: 187.901311 ms ± 12.134981 ms

Chi^2/NDF: 1.080462

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1076.14
NDf                       =          996
Edm                       =  6.52904e-06
NCalls                    =          125
source #                  =      137.669   +/-   3.15526      	 (limited)
Source lambda             =   0.00368889   +/-   0.000238235  	 (limited)
shift                     =      530.038   +/-   2.19774      	 (limited)
TH1.Print Name  = Decay_Curve_329KeV, Entries= 568091, Total sum= 567553


In file included from input_line_84:1:
In file included from /opt/anaconda3/envs/Sulfur/include/CPyCppyy/API.h:26:
In file included from /opt/anaconda3/envs/Sulfur/include/python3.11/Python.h:92:
/opt/anaconda3/envs/Sulfur/include/python3.11/compile.h:8:9: warning: 'Py_single_input' macro redefined [-Wmacro-redefined]
#define Py_single_input 256
        ^
:49:1: note: previous definition is here
   }
^


In [4]:
bounds = [[0,300], # source
          [1e-6,1],   #lamb
          [0,300]] #background shift
X0 = [120,
      .005,
      190]
bckgrndSub = rt.TCanvas("bckgrndSub","bckgrndSub_329KeV")
bckgrnd_329 = dtimeN.ProjectionX("Decay_Curve_329keV_bckgnd",320,324)
gamma_bckgrndsub = gamma_329 - bckgrnd_329
Decayp(gamma_bckgrndsub, bckgrndSub, X0,bounds)

Min X:  1
Max X:  1000.0
Initial p0 = 120.0
Initial p1 = 0.005
Initial p2 = 190.0
Parameter 0 (source #) limits: [c_double(0.0), c_double(300.0)]
Parameter 1 (Source lambda) limits: [c_double(1e-06), c_double(1.0)]
Parameter 2 (shift) limits: [c_double(0.0), c_double(300.0)]
Decay: 73.663738 ms ± 1.320780 ms

Chi^2/NDF: 35.438132

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      35296.4
NDf                       =          996
Edm                       =  5.04744e-06
NCalls                    =          150
source #                  =      117.838   +/-   1.87369      	 (limited)
Source lambda             =   0.00940961   +/-   0.000168713  	 (limited)
shift                     =      39.4763   +/-   0.262567     	 (limited)
TH1.Print Name  = Decay_Curve_329KeV, Entries= 86142, Total sum= 86142
